# Section V-A: OTEM on Multiple Qubits
**Fig. 13** — correct rate for QFT, GraphState, QuantumVolume on 3–7 qubit Aer simulators.
Three conditions: Fault-Free / Faulty (transient, swap_rate=1%) / Single-shot OTEM.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # qatg package lives in ../qatg/
sys.path.insert(0, os.path.abspath('.'))    # otem.py, fault_simulators.py live here

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.circuit.library import QFT, GraphState, QuantumVolume

from fault_simulators import (Transient_Qubit_Faulty_Backend,
                               get_counts_from_mem, get_em_counts_from_mem)

In [ ]:
def get_depolarizing_backend(error_rate_1q, error_rate_2q=None):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(error_rate_1q, 1), ['sx', 'rz', 'u'])
    if error_rate_2q:
        nm.add_all_qubit_quantum_error(depolarizing_error(error_rate_2q, 2), ['ecr', 'unitary'])
    return AerSimulator(noise_model=nm)

In [ ]:
faults     = np.load('data/faults.npy', allow_pickle=True)
ecr_faults = np.load('data/ecr_fault_list.npy', allow_pickle=True)
fault_id   = 6
sx_fault   = faults[fault_id]
ecr_fault1 = ecr_faults[1][fault_id]   # target qubit faulty
ecr_fault2 = ecr_faults[2][fault_id]   # control qubit faulty
ecr_fault3 = ecr_faults[3][fault_id]   # both qubits faulty
print(f"Loaded {len(faults)} SX fault models  |  using fault_id={fault_id}")

In [ ]:
nqc        = 29
test_datas = [QuantumCircuit().from_qasm_file(f'test_circuits/test_circuit_{i}.qasm')
              for i in range(nqc)]
# Use compressed index list matched to ibm_torino fault set (same as other experiments)
ltid       = [0, 1, 2, 3, 4, 5, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
test_datas = [test_datas[i] for i in ltid]
OT_CIRCUIT_ID = 5    # index within the compressed list (same as Comparison script)
print(f"Loaded {len(test_datas)} test circuits  |  OT_CIRCUIT_ID={OT_CIRCUIT_ID}")

In [ ]:
def correct_rate(counts, nqb=1):
    key = '0' * nqb
    tot = sum(counts.values())
    return counts.get(key, 0) / tot if tot > 0 else 0.0

## Sweep qubit counts 3–7
Uses `Transient_Qubit_Faulty_Backend` with per-qubit fault states (ECR + SX faults).
The **middle qubit** has `ft_to_ff_rate_magnification=1` so it spends ~50% of time
faulty, propagating errors through ECR gates and producing P_raw ≈ 0.3.

In [ ]:
SHOTS         = 50000   # lower for speed; paper uses 100 000
SWAP_RATE     = 0.005   # per-qubit fault toggle probability
BASIS_GATES   = ['sx', 'rz', 'ecr']
qubit_range   = range(3, 8)
circuit_types = ['QFT', 'GraphState', 'QuantumVolume']
results       = {ct: {'ff': [], 'faulty': [], 'otem': []} for ct in circuit_types}

ff_backend = get_depolarizing_backend(0.001, 0.005)


def make_transient_backend(nqb, coupling):
    """Build a Transient_Qubit_Faulty_Backend for `nqb` qubits.
    Middle qubit gets ft_to_ff_rate_magnification=1 (50 % faulty in steady state).
    All other qubits get magnification=1000 (almost always fault-free).
    """
    middle = nqb // 2
    ft_to_ff = [1000.0 if i != middle else 1.0 for i in range(nqb)]
    swap_rates = [SWAP_RATE] * nqb
    return Transient_Qubit_Faulty_Backend(
        sx_fault, ecr_fault1, ecr_fault2, ecr_fault3,
        good_backend=ff_backend,
        basis_gates=BASIS_GATES,
        num_qubits=nqb,
        coupling_map=coupling,
        swap_rate=swap_rates,
        ft_to_ff_rate_magnification=ft_to_ff,
    )


def make_ot_circuit(nqb):
    """Tensor the single-qubit test template (with measurements) across nqb qubits.
    Measurements must be kept so run_single_em can read the OT result.
    """
    tqc_base = test_datas[OT_CIRCUIT_ID]   # keep measurements intact
    qc_ot = tqc_base.copy()
    for _ in range(nqb - 1):
        qc_ot = tqc_base.tensor(qc_ot)
    return transpile(qc_ot, basis_gates=BASIS_GATES,
                     coupling_map=[[i, i+1] for i in range(nqb - 1)],
                     optimization_level=0)


def make_circuit(circ_cls, n, coupling):
    """Build benchmark circuit as forward ∘ inverse → ideal output |0…0⟩."""
    if circ_cls == 'QFT':
        gate = QFT(n)
        qc = QuantumCircuit(n)
        qc.append(gate, range(n)); qc.barrier()
        qc.append(gate.inverse(), range(n))
    elif circ_cls == 'GraphState':
        qc = GraphState([[1] * n] * n); qc.barrier()
        qc.append(qc.inverse(), range(n))
    else:  # QuantumVolume
        qc = QuantumVolume(n); qc.barrier()
        qc.append(qc.inverse(), range(n))
    qc.measure_all()
    return transpile(qc, basis_gates=BASIS_GATES, coupling_map=coupling,
                     optimization_level=1)


for nqb in qubit_range:
    print(f"\n--- {nqb} qubits ---")
    coupling = [[i, i + 1] for i in range(nqb - 1)]
    tf_b     = make_transient_backend(nqb, coupling)
    qc_ot    = make_ot_circuit(nqb)

    for ct in circuit_types:
        alg_qc = make_circuit(ct, nqb, coupling)

        # Fault-free
        ff_mem = ff_backend.run(alg_qc, shots=SHOTS, memory=True).result().get_memory()
        ff_r   = correct_rate(get_counts_from_mem(ff_mem), nqb)

        # Faulty (transient, no mitigation)
        ft_mem = tf_b.run(alg_qc, shots=SHOTS)[0]
        ft_r   = correct_rate(get_counts_from_mem(ft_mem), nqb)

        # Single-shot OTEM: interleave [OT_shot, alg_shot] × SHOTS
        em_mem = tf_b.run_single_em(qc_ot, alg_qc,
                                     ot_shots=1, alg_shots=1, repeat=SHOTS)
        em_cnt = get_em_counts_from_mem(em_mem, shots_ot=1, ot_pass_th=1, shots_alg=1)
        em_r   = correct_rate(em_cnt, nqb)

        results[ct]['ff'].append(ff_r)
        results[ct]['faulty'].append(ft_r)
        results[ct]['otem'].append(em_r)
        print(f"  {ct}: FF={ff_r:.3f}  Faulty={ft_r:.3f}  OTEM={em_r:.3f}")

## Plot — Fig. 13

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
lbls    = list(qubit_range)
mk = {'ff': ('o','tab:blue','Fault-Free'), 'faulty': ('s','tab:red','Faulty'),
      'otem': ('^','tab:green','OTEM')}
for ax, ct in zip(axes, circuit_types):
    for key, (m, c, lbl) in mk.items():
        ax.scatter(lbls, results[ct][key], marker=m, color=c, label=lbl, s=60)
    ax.set_xlabel('Number of Qubits'); ax.set_ylabel('Correct Rate')
    ax.set_title(f'({chr(97+circuit_types.index(ct))}) {ct}')
    ax.legend(fontsize=9); ax.set_ylim(0, 1); ax.set_xticks(lbls)
plt.suptitle('Fig. 13: Simulation result for different numbers of qubits')
plt.tight_layout()
plt.savefig('fig13_sim_multiqubit.pdf', bbox_inches='tight')
plt.show()